In [2]:
from pathlib import Path

import pandas as pd

project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "dataset").exists()), Path.cwd())
DATA_DIR = project_root / "dataset"

dfs = {
    "customers": pd.read_csv(DATA_DIR / "customers.csv", parse_dates=["signup_date"]),
    "geography": pd.read_csv(DATA_DIR / "geography.csv"),
    "inventory": pd.read_csv(DATA_DIR / "inventory.csv", parse_dates=["snapshot_date"]),
    "order_items": pd.read_csv(DATA_DIR / "order_items.csv"),
    "orders": pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"]),
    "payments": pd.read_csv(DATA_DIR / "payments.csv"),
    "products": pd.read_csv(DATA_DIR / "products.csv"),
    "promotions": pd.read_csv(DATA_DIR / "promotions.csv", parse_dates=["start_date", "end_date"]),
    "returns": pd.read_csv(DATA_DIR / "returns.csv", parse_dates=["return_date"]),
    "reviews": pd.read_csv(DATA_DIR / "reviews.csv", parse_dates=["review_date"]),
    "sales": pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["Date"]),
    "shipments": pd.read_csv(DATA_DIR / "shipments.csv", parse_dates=["ship_date", "delivery_date"]),
    "web_traffic": pd.read_csv(DATA_DIR / "web_traffic.csv", parse_dates=["date"]),
}

print(f"Loaded {len(dfs)} tables from {DATA_DIR}")


C:\Users\admin\AppData\Local\Temp\ipykernel_33444\3692375358.py:12: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  "order_items": pd.read_csv(DATA_DIR / "order_items.csv"),


Loaded 13 tables from c:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon\dataset


In [3]:
# Q1. Median inter-order gap

orders = dfs["orders"].copy()
orders = orders.sort_values(["customer_id", "order_date"])

orders["prev_order_date"] = orders.groupby("customer_id")["order_date"].shift(1)
orders["gap_days"] = (orders["order_date"] - orders["prev_order_date"]).dt.days

multi_order_customers = orders["customer_id"].value_counts()
multi_order_customers = multi_order_customers[multi_order_customers > 1].index

q1_df = orders[orders["customer_id"].isin(multi_order_customers)].copy()
gap_series = q1_df["gap_days"].dropna()

median_gap = gap_series.median()

print("Median gap:", median_gap)
print(gap_series.describe())

Median gap: 144.0
count    556699.000000
mean        285.592509
std         389.691558
min           0.000000
25%          46.000000
50%         144.000000
75%         357.000000
max        3785.000000
Name: gap_days, dtype: float64


In [4]:
# Q2
# Segment nào có gross margin ratio trung bình cao nhất?
# (price - cogs) / price
# =========================
products = dfs["products"].copy()
products["gross_margin_ratio"] = (products["price"] - products["cogs"]) / products["price"]

segment_margin = (
    products.groupby("segment", as_index=False)["gross_margin_ratio"]
    .mean()
    .sort_values("gross_margin_ratio", ascending=False)
)

print(segment_margin)

top_segment = segment_margin.iloc[0]["segment"]
q2_options = {
    "A": "Premium",
    "B": "Performance",
    "C": "Activewear",
    "D": "Standard",
}


       segment  gross_margin_ratio
6     Standard            0.313442
5      Premium            0.285377
1  All-weather            0.284176
0   Activewear            0.265600
4  Performance            0.263650
2     Balanced            0.258038
7       Trendy            0.240758
3     Everyday            0.236343


In [5]:
# Q3
# Trong returns nối với products theo product_id,
# với category = Streetwear, return_reason nào nhiều nhất?
# =========================
q3_df = (
    dfs["returns"][["return_id", "order_id", "product_id", "return_reason"]]
    .merge(dfs["products"][["product_id", "category"]], on="product_id", how="left")
)

q3_df = q3_df[q3_df["category"] == "Streetwear"]

reason_counts = q3_df["return_reason"].value_counts(dropna=False)
print(reason_counts)

top_reason = reason_counts.idxmax()
q3_options = {
    "A": "defective",
    "B": "wrong_size",
    "C": "changed_mind",
    "D": "not_as_described",
}


return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


In [6]:
# Q4
# traffic_source nào có average bounce_rate thấp nhất?
# =========================

web = dfs["web_traffic"].copy()

source_bounce = (
    web.groupby("traffic_source", as_index=False)["bounce_rate"]
    .mean()
    .sort_values("bounce_rate", ascending=True)
)

print(source_bounce)

best_source = source_bounce.iloc[0]["traffic_source"]
q4_options = {
    "A": "organic_search",
    "B": "paid_search",
    "C": "email_campaign",
    "D": "social_media",
}

   traffic_source  bounce_rate
1  email_campaign     0.004458
5    social_media     0.004476
3     paid_search     0.004478
4        referral     0.004499
2  organic_search     0.004504
0          direct     0.004511


In [7]:
# Q5
# % dòng order_items có promo_id không null
# =========================

oi = dfs["order_items"].copy()
promo_rate = float(oi["promo_id"].notna().mean() * 100)

print(f"Promo usage rate: {promo_rate:.4f}%")

q5_options = {"A": 12, "B": 25, "C": 39, "D": 54}


Promo usage rate: 38.6635%


In [8]:
# Q6
# age_group nào có avg số đơn / customer cao nhất
# chỉ xét customers có age_group != null
# =========================

customers = dfs["customers"].copy()
orders = dfs["orders"].copy()

cust_non_null = customers[customers["age_group"].notna()].copy()

orders_per_customer = (
    orders.groupby("customer_id", as_index=False)
    .size()
    .rename(columns={"size": "num_orders"})
)

q6_df = cust_non_null.merge(orders_per_customer, on="customer_id", how="left")
q6_df["num_orders"] = q6_df["num_orders"].fillna(0)

age_group_stats = (
    q6_df.groupby("age_group", as_index=False)["num_orders"]
    .mean()
    .sort_values("num_orders", ascending=False)
)

print(age_group_stats)

top_age_group = age_group_stats.iloc[0]["age_group"]
q6_options = {
    "A": "55+",
    "B": "25–34",
    "C": "35–44",
    "D": "45–54",
}
# normalize dash variants
top_age_group_norm = str(top_age_group).replace("-", "–")


  age_group  num_orders
4       55+    5.406851
3     45-54    5.357241
2     35-44    5.337343
1     25-34    5.245226
0     18-24    5.226656


In [11]:
#Q7
q7_df = (
    dfs["order_items"][["order_id", "product_id", "quantity", "unit_price"]]
    .merge(dfs["orders"][["order_id", "zip"]], on="order_id", how="left")
    .merge(dfs["geography"][["zip", "region"]], on="zip", how="left")
)

q7_df["line_revenue"] = q7_df["quantity"] * q7_df["unit_price"]

region_revenue = (
    q7_df.groupby("region", as_index=False)["line_revenue"]
    .sum()
    .sort_values("line_revenue", ascending=False)
)

print(region_revenue)

# If very close, choose D
top1 = region_revenue.iloc[0]
top2 = region_revenue.iloc[1]
relative_gap = (top1["line_revenue"] - top2["line_revenue"]) / top1["line_revenue"]

q7_options = {
    "A": "West",
    "B": "Central",
    "C": "East",
    "D": "Approximately equal",
}

if relative_gap < 0.01:  # adjustable threshold for "xấp xỉ bằng nhau"
    q7_answer = "D"
    q7_result = "Approximately equal"
else:
    q7_result = top1["region"]
    q7_answer = {"West": "A", "Central": "B", "East": "C"}.get(q7_result, None)

print("Top region:", q7_result)
print("Relative gap vs 2nd:", relative_gap)
print("Chosen answer:", q7_answer)

    region  line_revenue
1     East  7.637533e+09
0  Central  4.941908e+09
2     West  3.851035e+09
Top region: East
Relative gap vs 2nd: 0.35294437599201056
Chosen answer: C


In [13]:
# Q8
# Trong orders có status = cancelled, payment_method nào nhiều nhất?
# =========================

cancelled = dfs["orders"].copy()
cancelled = cancelled[cancelled["order_status"] == "cancelled"]

payment_counts = cancelled["payment_method"].value_counts(dropna=False)
print(payment_counts)

top_payment = payment_counts.idxmax()
q8_options = {
    "A": "credit_card",
    "B": "cod",
    "C": "paypal",
    "D": "bank_transfer",
}


payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


In [20]:
# Q9. Highest return rate by size
# Definition:
# return rate = number of records in returns / number of rows in order_items
# join with products by product_id to get size
# =========================

prod_size = dfs["products"][["product_id", "size"]].copy()

# Denominator: number of order_items rows by size
order_items_size = (
    dfs["order_items"][["order_id", "product_id"]]
    .merge(prod_size, on="product_id", how="left")
)

denom = (
    order_items_size.groupby("size")
    .size()
    .reset_index(name="order_item_rows")
)

# Numerator: number of return records by size
returns_size = (
    dfs["returns"][["return_id", "product_id"]]
    .merge(prod_size, on="product_id", how="left")
)

numer = (
    returns_size.groupby("size")
    .size()
    .reset_index(name="return_rows")
)

# Merge numerator and denominator
q9_stats = denom.merge(numer, on="size", how="left")
q9_stats["return_rows"] = q9_stats["return_rows"].fillna(0)

# Keep only the four sizes mentioned in the question
q9_stats = q9_stats[q9_stats["size"].isin(["S", "M", "L", "XL"])].copy()

q9_stats["return_rate"] = q9_stats["return_rows"] / q9_stats["order_item_rows"]
q9_stats = q9_stats.sort_values("return_rate", ascending=False)

print(q9_stats)

top_size = q9_stats.iloc[0]["size"]
q9_options = {"A": "S", "B": "M", "C": "L", "D": "XL"}


  size  order_item_rows  return_rows  return_rate
2    S           172042         9723     0.056515
0    L           173174         9741     0.056250
1    M           176428         9820     0.055660
3   XL           193025        10655     0.055200


In [16]:
# Q10
# installments nào có avg payment_value cao nhất?
# =========================

payments = dfs["payments"].copy()

installment_stats = (
    payments.groupby("installments", as_index=False)["payment_value"]
    .mean()
    .sort_values("payment_value", ascending=False)
)

print(installment_stats)

best_installments = int(installment_stats.iloc[0]["installments"])
q10_options = {"A": 1, "B": 3, "C": 6, "D": 12}


   installments  payment_value
3             6   24446.654403
2             3   24399.635486
4            12   24245.772694
0             1   24113.274166
1             2     708.473729
